In [1]:
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt
import pandas as pd
from fonts_config import set_computer_modern, truncate_colormap
set_computer_modern()
import matplotlib as mpl
mpl.rcParams['axes.unicode_minus'] = False
import matplotlib.lines as mlines
from matplotlib.patches import Rectangle, Patch
import glob
import os
from matplotlib.legend_handler import HandlerTuple
import matplotlib.ticker as ticker

from matplotlib import cm
import matplotlib.colors as mcolors


In [2]:
# ----------------------------------------------------------------------------
# Paths
# ----------------------------------------------------------------------------
sims=xr.open_dataset("../output/timeseries_from1D.nc")
temp=xr.open_dataset("../output/timeseries_from2D.nc")
leger=xr.open_dataset("../../FesmData/Leger2024_PaleoGris/output/regions_area_paleogris.nc")
scores = xr.open_dataset("../scoring/scores/scores_final.nc")
n_best=2572
best=sims.sel(sim=n_best)
T_srf_best=temp.sel(sim=n_best)
# Observations
A_obs=1.7
V_obs=7.4
#scores
S = xr.open_dataset("../scoring/scores/scores_final.nc")

In [3]:
s_values = S.S.values 
colors = ["#D9DAB3", "#3C9D9F", "#002665"]
nodes = [0, 0.3, 1.0]
cmap = mcolors.LinearSegmentedColormap.from_list("custom_warm", list(zip(nodes, colors)))
sorted_sim_indices = S.sim.sortby(S.S).values
norm = mcolors.Normalize(vmin=s_values.min()*100, vmax=s_values.max()*100)

fig, axs = plt.subplots(3, 1, figsize=(5, 12),sharex=True)

for index in sorted_sim_indices:
    sim = sims.sel(sim=index)
    t_sim = temp.sel(sim=index) 
    color_s = cmap(norm(S.S.sel(sim=index).values*100))
    
    # Plot Temp
    axs[0].plot(t_sim.time/1e3, t_sim.T_ann, color=color_s, alpha=0.3, linewidth=0.5, zorder=1)
    axs[0].plot(t_sim.time/1e3, t_sim.T_sum, color=color_s, alpha=0.3, linestyle='solid', linewidth=0.5, zorder=1)
    
    # Plot Area
    axs[1].plot(sim.time/1e3, sim.A_ice_g, color=color_s, alpha=0.3, linewidth=0.5, zorder=1)
    
    # Plot Volume
    axs[2].plot(sim.time/1e3, sim.V_sle, color=color_s, alpha=0.3, linewidth=0.5, zorder=1)


axs[0].plot(T_srf_best.time/1e3, T_srf_best.T_ann, color='black', linewidth=1.2, zorder=10,label="Annual")
axs[0].plot(T_srf_best.time/1e3, T_srf_best.T_sum, color='black', linestyle='dotted', linewidth=2, zorder=10,label="Summer")
axs[0].legend(frameon=False, loc='lower right')
axs[1].plot(best.time/1e3, best.A_ice_g, color='black', linewidth=1.5, zorder=10)
axs[2].plot(best.time/1e3, best.V_sle, color='black', linewidth=1.5, zorder=10)

present = axs[1].axhline(y=A_obs, color='#DB74E0', linestyle='-', alpha=0.8, linewidth=2, zorder=2)
rect = Rectangle((-19, 2.94), 3, 0.2, facecolor="none", edgecolor='red', linewidth=2,alpha=0.8, zorder=4)
axs[1].add_patch(rect)
paleogris = axs[1].errorbar(-leger["time"][1:] * 1e-3, leger.tot[1:], xerr=leger.time_err[1:]*1e-3, 
                            fmt='o', ecolor='black', markersize=5, markerfacecolor='red', 
                            markeredgecolor='black', markeredgewidth=0.8, capsize=2, zorder=10)

axs[2].axhline(y=V_obs, color="#DB74E0", linestyle='-', alpha=0.8, linewidth=2, zorder=2)
# time periods
axs[2].text(-20.5, 5, "LGM", ha="center", va="center")
axs[2].text(-13.8, 5, "BA",  ha="center", va="center")
axs[2].text(-12.2, 5, "YD",  ha="center", va="center")
axs[2].text(-8, 5, "HTM",  ha="center", va="center")

# axis
axs[0].set_ylabel('Surface temperature (K)')
axs[1].set_ylabel('A (million km$^2$)')
axs[2].set_ylabel('SLE (m)')
axs[2].set_xlabel('Time (kyr ago)')
for i, ax in enumerate(axs):
    ax.grid(alpha=0.2, zorder=0)
    ax.set_xlim(-22, 0)
    ax.text(0.9, 0.92, f'({chr(97+i)})', transform=ax.transAxes)
    ax.axvspan(-22, -19, color="#e2e2e2", alpha=0.5,zorder=0) 
    ax.axvspan(-19, -14.8, color="white", alpha=0.5,zorder=0)  
    ax.axvspan(-14.6, -12.8, color="#e2e2e2", alpha=0.5,zorder=0)   
    ax.axvspan(-12.8, -11.7, color="#a8a8a8", alpha=0.5,zorder=0)  
    ax.axvspan(-10, -6, color="#e2e2e2", alpha=0.5,zorder=0) 

# legend
h_list, l_list = [], []
best_line = mlines.Line2D([], [], color='black', linewidth=1.5)
ensemble_sample = mlines.Line2D([], [], color=cmap(0.5), alpha=0.5, linewidth=1)

h_list = [best_line, present, rect, paleogris]
l_list = [f'Best simulation', 
          'PD observations (Morlighem et al. 2014)', 'local LGM (Leger et al., 2024)', 
          'PaleoGrIS (Leger et al., 2024)']

leg = fig.legend(h_list, l_list, frameon=False, loc='upper left', ncol=1, bbox_to_anchor=(0.1, 0.99))
leg.get_frame().set_linewidth(0)

cbar_ax = fig.add_axes([0.12, 0.875, 0.4, 0.015])
fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax)
cb = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax, orientation='horizontal')
cb.set_label('Global score S (x10²)', labelpad=-30, x=1.5)

plt.tight_layout()
plt.subplots_adjust(top=0.85) 

fig.savefig(f"../figs_final/fig2_{n_best}ensemble_timeseries.pdf", dpi=300)
plt.close()

/tmp/ipykernel_2852328/1529002359.py:79: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
